# Demo: Klasifikasi Sampah (Organic vs Recyclable)

Notebook ini menyiapkan **training** model CNN sederhana dan **Flask API** untuk prediksi + penjelasan analisis.

Target demo: cepat, jelas, dan cocok untuk presentasi kampus.

## 1) Struktur Dataset

Script ini mendukung 2 pola folder kelas:

- Pola requirement: `dataset/organic/` dan `dataset/recyclable/`
- Pola dataset Anda: `DATASET/TRAIN/O/` dan `DATASET/TRAIN/R/` (O=Organic, R=Recyclable)

Default training akan mencari folder `DATASET/TRAIN` otomatis.

In [1]:
# (Opsional) cek contoh file gambar dari dataset Anda
from pathlib import Path

candidates = [
    Path('DATASET') / 'TRAIN' / 'O',
    Path('DATASET') / 'TRAIN' / 'R',
    Path('DATASET') / 'DATASET' / 'TRAIN' / 'O',
    Path('dataset') / 'organic',
]

for c in candidates:
    if c.exists():
        images = list(c.glob('**/*.*'))
        print('Folder:', c, '| jumlah file:', len(images))
        if images:
            print('Contoh file:', images[0])
        break

Folder: DATASET\TRAIN\O | jumlah file: 12565
Contoh file: DATASET\TRAIN\O\O_1.jpg


## 2) Install Dependency

Jika Anda pakai virtualenv/conda, aktifkan environment dulu, lalu install requirements.

In [2]:
# Jalankan di terminal notebook (untuk Jupyter lokal/Colab)
# !pip install -r requirements.txt
print('Silakan jalankan: pip install -r requirements.txt')

Silakan jalankan: pip install -r requirements.txt


## 3) Training Model (CNN sederhana)

Requirement yang dipenuhi di `train.py`:

- `ImageDataGenerator` + `rescale=1./255`
- Resize ke **150×150**
- Binary classification (`sigmoid`)
- CNN sederhana (Conv2D + MaxPooling + Dense)
- Epoch default: **8** (bisa diubah 5–10)
- Output: `model.h5` (+ `class_indices.json` untuk mapping kelas)

In [3]:
# Jalankan training
# Default akan cari DATASET/TRAIN
# !python train.py --epochs 8
# Atau spesifikkan folder:
# !python train.py --train_dir DATASET/TRAIN --epochs 8
print('Contoh: python train.py --train_dir DATASET/TRAIN --epochs 8')

Contoh: python train.py --train_dir DATASET/TRAIN --epochs 8


## 4) Jalankan Flask API

Endpoint: **POST /predict**

- Input: upload file gambar (multipart/form-data) dengan key `image`
- Output: JSON berisi `prediction`, `confidence`, dan `analysis`
- Bonus presentasi: jika `confidence < 0.6` maka `prediction = "Low confidence prediction"`

In [4]:
# Jalankan API
# !python app.py
print('Jalankan: python app.py')

Jalankan: python app.py


## 5) Contoh Request (cURL)

Ganti path gambar sesuai file yang ada di dataset Anda. Contoh (Windows PowerShell):

```bash
curl -X POST http://127.0.0.1:5000/predict -F "image=@DATASET\TEST\O\contoh.jpg"
```

## 6) Integrasi Laravel (gambaran singkat)

Di Laravel, Anda bisa kirim gambar ke Flask memakai `Http::attach(...)`.

Contoh (controller):

```php
use Illuminate\Support\Facades\Http;

$response = Http::attach(
    'image',
    file_get_contents($request->file('image')->getRealPath()),
    $request->file('image')->getClientOriginalName()
)->post('http://127.0.0.1:5000/predict');

return $response->json();
```